# 02 — Preprocessing & Augmentation

We turn raw, inconsistent fundus photos into clean, standardized model inputs, and we set up
**gentle** random augmentations for training. This notebook shows each step **before/after** so
you can decide what to enable in `config.yaml`.

> ⚠️ Research/education only — not a medical device.


In [ ]:
import sys, os; sys.path.append(os.path.abspath('..'))
import numpy as np, cv2, matplotlib.pyplot as plt
from src.utils import load_config, set_seed
from src.preprocess import step_by_step, build_preprocess
from src.data import read_labels, generate_synthetic_dataset
cfg=load_config('../config.yaml'); set_seed(cfg['project']['seed'])
FIG='../reports/figures'; os.makedirs(FIG, exist_ok=True)


In [ ]:
# load a real image if available, else synthetic demo
try:
    df=read_labels(cfg)
except FileNotFoundError:
    generate_synthetic_dataset('../data/_synthetic_demo/aptos2019', seed=cfg['project']['seed'])
    dcfg={**cfg,'paths':{**cfg['paths'],'train_csv':'../data/_synthetic_demo/aptos2019/train.csv','train_images':'../data/_synthetic_demo/aptos2019/train_images'}}
    df=read_labels(dcfg)
row=df[df.diagnosis==df.diagnosis.max()].iloc[0]
img=cv2.cvtColor(cv2.imread(row['path']),cv2.COLOR_BGR2RGB)
print('image shape:', img.shape)


## 1. Deterministic preprocessing — step by step
**FOV crop** removes black borders so the retina fills the frame. **Ben-Graham** subtracts a
blurred local average to neutralize camera color/lighting and make lesions pop. **CLAHE** is an
optional local-contrast boost (off by default — it overlaps with Ben-Graham). **Resize** brings
everything to the model's input size (224). ImageNet **normalization** is applied later, in the
augmentation transforms, so train and eval share one definition.


In [ ]:
steps=step_by_step(img,cfg)
order=['original','crop_fov','ben_graham','clahe','resize']
titles=['Original','1. FOV crop','2. Ben-Graham','3. CLAHE','4. Resize 224']
fig,axes=plt.subplots(1,5,figsize=(16,3.4))
for ax,k,t in zip(axes,order,titles): ax.imshow(steps[k]); ax.set_title(t,fontsize=10); ax.axis('off')
fig.tight_layout(); fig.savefig(f'{FIG}/preprocess_steps.png',dpi=140,bbox_inches='tight'); plt.show()


## 2. The composed preprocess function
`build_preprocess(cfg)` returns ONE callable that applies the enabled steps in order. This is the
exact function passed to the Dataset, so training, validation, test, and the live app all clean
images identically.


In [ ]:
pre=build_preprocess(cfg)
out=pre(img)
print('preprocessed ->', out.shape, out.dtype)
plt.imshow(out); plt.title('build_preprocess output'); plt.axis('off'); plt.show()


## 3. Training augmentations (gentle on purpose)
Augmentation runs **only on training data**, re-rolled each epoch. We use horizontal/vertical
flips and small rotations (a retina has no canonical orientation), plus mild brightness/contrast
and a small shift/zoom to mimic camera variation.

**Why gentle?** DR grade depends on tiny lesions a few pixels wide. Aggressive crops, heavy zoom,
or strong warps can erase or fabricate that evidence — so we augment for robustness without
destroying the signal. Validation/test get normalize + to-tensor only (no randomness).


In [ ]:
# Visualize a few augmentations. Uses Albumentations if installed (from requirements.txt).
try:
    from src.augment import build_train_transforms, denormalize
    tf=build_train_transforms(cfg)
    base=pre(img)
    fig,axes=plt.subplots(1,6,figsize=(16,3))
    axes[0].imshow(base); axes[0].set_title('preprocessed'); axes[0].axis('off')
    for i in range(1,6):
        t=tf(image=base)['image']; axes[i].imshow(denormalize(t,cfg)); axes[i].set_title(f'aug {i}'); axes[i].axis('off')
    fig.tight_layout(); fig.savefig(f'{FIG}/augmentation_examples.png',dpi=140,bbox_inches='tight'); plt.show()
except ModuleNotFoundError:
    print('Install albumentations (in requirements.txt) to run live augmentation; a precomputed example is saved in reports/figures/augmentation_examples.png')


## 4. Handling class imbalance
From EDA we know the data is dominated by 'No DR'. Two complementary tools (configured in
`config.yaml → imbalance`) are implemented in `src/data.py`:

1. **Class-weighted loss** (`compute_class_weights`): scale CrossEntropy by inverse class
   frequency, so rare grades contribute as much to the loss as the common ones.
2. **WeightedRandomSampler** (`make_weighted_sampler`): oversample rare grades when drawing
   training batches, so each batch is roughly balanced.

A third option, **focal loss**, down-weights easy, confidently-correct examples to focus on hard
ones; we expose `focal_gamma` and wire it into the training loss during training. Start with
weighted loss alone; combining all three can over-correct.


In [ ]:
from src.data import compute_class_weights, make_splits, class_distribution
try:
    tr,va,te=make_splits(df,cfg)
    w=compute_class_weights(tr, cfg['classes']['num_classes'])
    print('train class counts:', class_distribution(tr,5).tolist())
    print('class weights (mean=1):', [round(float(x),3) for x in w])
    print('-> rare grades get larger weights, common grades smaller')
except ModuleNotFoundError:
    print('scikit-learn / torch needed here; available on Colab via requirements.txt')


## Summary
We standardize every image (FOV crop → Ben-Graham → optional CLAHE → resize → ImageNet
normalize) and augment training images **gently** (flips, small rotations, mild brightness).
Class imbalance is handled via weighted loss and/or a balanced sampler. All choices live in
`config.yaml`, and the WHY behind each is in `docs/pipeline.md`.

_Next: the custom CNN built from scratch._
